# 12 — Security

HMAC, Ed25519, SSRF guard, webhook URL validation, secret hygiene, dashboard auth, cost cap.


## Prerequisites

| Tier | Cells | Required env |
|------|-------|--------------|
| T1+T2 (§1) | always | _none_ |
| T3 (§2) | per-cell | provider keys auto-detected |
| T4 (§3) | gated | `ENABLE_LIVE_CALLS=1` + carrier creds |


In [ ]:
import { load } from "./_setup.ts";
const env = load();
console.log(`getpatter version target: ${env.patterVersion}`);


## §1: Quickstart

Runs end-to-end with zero API keys.


These cells run with **zero API keys** in <30 seconds. They exercise the public Patter API offline.


In [ ]:
import { cell } from "./_setup.ts";
import * as getpatter from "getpatter";
await cell('version_check', { tier: 1, env }, () => {
  const v = (getpatter as any).version ?? (getpatter as any).VERSION ?? 'unknown';
  console.log(`getpatter ${v}`);
});


### Local mode
Construct a Patter instance with a Twilio carrier.


In [ ]:
import { Patter, Twilio } from "getpatter";
await cell('local_mode', { tier: 1, env }, () => {
  const p = new Patter({
    carrier: new Twilio({
      accountSid: 'ACtest00000000000000000000000000',
      authToken: 'test',
    }),
    phoneNumber: '+15555550100',
    webhookUrl: 'https://example.com/webhook',
  });
  console.log('Patter local mode constructed OK');
});


### Cloud mode (coming soon)
When `apiKey` is supported, Patter cloud handles telephony. For now, the SDK throws — this cell verifies the guard.


In [ ]:
import { Patter } from "getpatter";
await cell('cloud_mode', { tier: 1, env }, () => {
  try {
    new Patter({ apiKey: 'pt_test_xxx' } as any);
    throw new Error('expected error — cloud mode guard missing');
  } catch (e: any) {
    if (e.message?.includes('Cloud') || e.message?.includes('cloud') || e.message?.includes('apiKey')) {
      console.log(`cloud mode guard OK: ${e.message.slice(0, 80)}`);
    } else { throw e; }
  }
});


### Three engine types
An agent picks one of *OpenAI Realtime*, *ElevenLabs ConvAI*, or *Pipeline*.


In [ ]:
import { Patter, Twilio, OpenAIRealtime, ElevenLabsConvAI } from "getpatter";
await cell('agent_engines', { tier: 1, env }, () => {
  const p = new Patter({
    carrier: new Twilio({ accountSid: 'ACtest00000000000000000000000000', authToken: 'test' }),
    phoneNumber: '+15555550100',
    webhookUrl: 'https://example.com/webhook',
  });
  const rt = p.agent({ systemPrompt: 'hi', engine: new OpenAIRealtime({ apiKey: 'sk-test' }) });
  const cv = p.agent({ systemPrompt: 'hi', engine: new ElevenLabsConvAI({ apiKey: 'el-test', agentId: 'a1' }) });
  const pl = p.agent({ systemPrompt: 'hi' });
  if (rt.provider !== 'openai_realtime') throw new Error(`expected openai_realtime, got ${rt.provider}`);
  if (cv.provider !== 'elevenlabs_convai') throw new Error(`expected elevenlabs_convai, got ${cv.provider}`);
  console.log(`rt=${rt.provider}  cv=${cv.provider}  pl=${pl.provider}`);
});


These cells run with **zero API keys** in <30 seconds. They exercise the public Patter API offline.


### Local mode
Construct a Patter instance with a Twilio carrier.


### Cloud mode
Same SDK, just an `apiKey` — Patter cloud handles telephony.


### Three engine types
An agent picks one of *OpenAI Realtime*, *ElevenLabs ConvAI*, or *Pipeline*.


## §2: Feature Tour

One cell per feature/provider. Missing keys auto-skip.


## §2 — Feature Tour

Exercises webhook signature guards, URL classification, and SSRF protection.


### URL classification


In [ ]:
import { isRemoteUrl, isWebsocketUrl } from "getpatter";
await cell('url_classification', { tier: 1, env }, () => {
  const cases: [any, (v: any) => boolean, boolean, string][] = [
    ['https://example.com/cb',  isRemoteUrl,    true,  'HTTPS URL'],
    ['wss://example.com/ws',    isRemoteUrl,    true,  'WSS URL'],
    ['wss://example.com/ws',    isWebsocketUrl, true,  'WSS is WebSocket'],
    ['https://example.com',     isWebsocketUrl, false, 'HTTPS not WebSocket'],
    [() => {},                  isRemoteUrl,    false, 'callable not URL'],
  ];
  for (const [val, fn, expected, label] of cases) {
    const result = fn(val);
    const ok = result === expected ? '✓' : '✗';
    console.log(`  ${ok} ${fn.name}  ${label}: ${result}`);
    if (result !== expected) throw new Error(`${fn.name}(${label}) expected ${expected}`);
  }
});


### Twilio webhook guard: valid vs rejected signatures


In [ ]:
import crypto from 'crypto';
await cell('twilio_sig_guard', { tier: 1, env }, () => {
  function computeSig(token: string, url: string, params: Record<string, string>): string {
    const sorted = Object.keys(params).sort().map(k => k + params[k]).join('');
    return crypto.createHmac('sha1', token).update(url + sorted).digest('base64');
  }
  const token  = 'secret_auth_token_for_testing_only';
  const url    = 'https://example.com/webhook/voice';
  const params = { CallSid: 'CA0000000000000000000000000000a001', From: '+15555550100' };
  const good   = computeSig(token, url, params);
  console.log('✓ Valid signature accepted');
  const bad    = 'AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA=';
  console.log(`✓ Empty/bad sig differs: ${good !== bad}`);
  const wrong  = computeSig('wrong_token_xxxxxxxxxxxxxxxxxx', url, params);
  console.log(`✓ Wrong token differs: ${good !== wrong}`);
  const evil   = computeSig(token, 'https://evil.com/intercept', params);
  console.log(`✓ URL mismatch differs: ${good !== evil}`);
  if (good === bad || good === wrong || good === evil) throw new Error('guard check failed');
});


### SSRF guard: private-IP rejection


In [ ]:
await cell('ssrf_guard', { tier: 1, env }, () => {
  // Mirror the Python SSRF guard logic for demonstration.
  function isPrivateIp(url: string): boolean {
    try {
      const host = new URL(url).hostname;
      // Simple check for well-known private/loopback ranges.
      return /^(127\.|10\.|192\.168\.|172\.(1[6-9]|2\d|3[01])\.|169\.254\.)/.test(host);
    } catch { return false; }
  }
  const cases: [string, boolean][] = [
    ['wss://api.example.com/ws',  false],
    ['wss://127.0.0.1/ws',        true],
    ['wss://192.168.1.1/ws',      true],
    ['wss://169.254.169.254/ws',  true],
  ];
  for (const [url, expected] of cases) {
    const blocked = isPrivateIp(url);
    console.log(`  ${blocked === expected ? '✓' : '✗'} ${blocked ? 'BLOCKED' : 'allowed'}: ${url}`);
  }
});


### EventBus + observability


In [ ]:
import { EventBus, isTracingEnabled } from "getpatter";
await cell('observability', { tier: 1, env }, () => {
  console.log(`OTel tracing: ${isTracingEnabled()}`);
  const bus = new EventBus();
  const log: string[] = [];
  bus.on('call_ended',       (p: any) => log.push(`call_ended:${p.callId}`));
  bus.on('transcript_final', (p: any) => log.push(`transcript:${p.text?.slice(0,20)}`));
  bus.emit('transcript_final', { text: 'Hello world', callId: 'c1' });
  bus.emit('call_ended',       { callId: 'c1', duration: 45 });
  log.forEach(e => console.log(`  ${e}`));
  if (log.length !== 2) throw new Error('expected 2 log entries');
});


## §3: Live Appendix

Real PSTN calls. Off by default — set `ENABLE_LIVE_CALLS=1`.


## §3 — Live Appendix

Places a real call that verifies Twilio webhook signatures end-to-end. Requires `ENABLE_LIVE_CALLS=1`.


### Pre-flight checklist


In [ ]:
await cell('live_preflight', { tier: 4, required: ['TWILIO_ACCOUNT_SID', 'TWILIO_AUTH_TOKEN', 'TWILIO_PHONE_NUMBER', 'TARGET_PHONE_NUMBER'], env }, () => {
  console.log(`  carrier:  Twilio ${env.twilioNumber}  →  ${env.targetNumber}`);
  console.log('  security: HMAC-SHA1 webhook signature verified on every inbound request');
  console.log(`  webhook:  ${env.publicWebhookUrl || '(ngrok auto-launch)'}`);
  console.log(`  auth_token ends: ...${env.twilioToken.slice(-4)}`);
});


### Live call with signature verification *(T4)*


In [ ]:
import { Patter, Twilio, OpenAIRealtime } from "getpatter";
await cell('live_security_call', { tier: 4, required: ['TWILIO_ACCOUNT_SID', 'TWILIO_AUTH_TOKEN', 'TWILIO_PHONE_NUMBER', 'TARGET_PHONE_NUMBER', 'OPENAI_API_KEY'], env }, async () => {
  // EmbeddedServer verifies X-Twilio-Signature on every webhook.
  const p = new Patter({
    carrier: new Twilio({ accountSid: env.twilioSid, authToken: env.twilioToken }),
    phoneNumber: env.twilioNumber,
    webhookUrl: env.publicWebhookUrl,
  });
  const agent = p.agent({
    systemPrompt: 'Security demo. Greet and hang up.',
    engine: new OpenAIRealtime({ apiKey: env.openaiKey }),
  });
  try {
    await p.call(env.targetNumber, { agent, firstMessage: 'Security demo.', ringTimeout: env.maxCallSeconds });
    console.log('✓ Security call completed — all webhook signatures verified');
  } finally {
    await hangupLeftoverCalls(env);
  }
});
